# Введение в MapReduce модель на Python


In [1]:
from typing import NamedTuple
from typing import Iterator


In [2]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)


Модель элемента данных

In [3]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str


In [4]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]


Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [5]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]


In [6]:
list(RECORDREADER())


[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [7]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element


In [8]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output)
map_output


[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()


In [10]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output


[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [11]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output


[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [12]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))


[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных. 

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [13]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))


## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*
 
mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL 

In [14]:
from typing import NamedTuple
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str
    
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)
 
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output


[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication 

In [15]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4)

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])
 
def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])
      
output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output


[(0, np.float64(2.0911942874576317)),
 (1, np.float64(2.0911942874576317)),
 (2, np.float64(2.0911942874576317)),
 (3, np.float64(2.0911942874576317)),
 (4, np.float64(2.0911942874576317))]

## Inverted index 

In [16]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)
      
def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)
 
def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output


[('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [17]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output


[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [18]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()
      
def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]
 
def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers
  
def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER)
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)
  
  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs


## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*
 
flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount 

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)
      
  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)
  
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None) 
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output


56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('it', 18)]),
 (1, [('banana', 2), ('is', 18), ('what', 10)])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for value in split:
        yield (value, None)
      
  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])
    
def MAP(value:int, _):
  yield (value, None)
  
def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)
  
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output


30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.10067388302139202)),
   (None, np.float64(0.11560388736758287)),
   (None, np.float64(0.1235171793483879)),
   (None, np.float64(0.20878584217856366)),
   (None, np.float64(0.2867967698188246)),
   (None, np.float64(0.2973124949282978)),
   (None, np.float64(0.3871781019697269)),
   (None, np.float64(0.3877857901585332)),
   (None, np.float64(0.4287990099941372)),
   (None, np.float64(0.44675424817504394)),
   (None, np.float64(0.456289639708255)),
   (None, np.float64(0.46393930255160376)),
   (None, np.float64(0.46575787148896197)),
   (None, np.float64(0.4805709994534555))]),
 (1,
  [(None, np.float64(0.510996940319724)),
   (None, np.float64(0.5224589533172815)),
   (None, np.float64(0.536156730048135)),
   (None, np.float64(0.5418265135574057)),
   (None, np.float64(0.5514498744894829)),
   (None, np.float64(0.5604009811596566)),
   (None, np.float64(0.6037859176730935)),
   (None, np.float64(0.6150207516728949)),
   (None, np.float64(0.6487175190420272

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [21]:
numbers = [3, 14, -5, 27, 11, 27, 8, 19]

def RECORDREADER():
  for idx, value in enumerate(numbers):
    yield (idx, value)

def MAP(_, value):
  yield ("max", value)

def REDUCE(key, values):
  yield (key, max(values))

max_result = list(MapReduce(RECORDREADER, MAP, REDUCE))
max_result


[('max', 27)]

### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [22]:
numbers = [3, 14, -5, 27, 11, 27, 8, 19]

def RECORDREADER():
  for idx, value in enumerate(numbers):
    yield (idx, value)

def MAP(_, value):
  yield ("avg", (value, 1))

def REDUCE(key, values):
  total_sum = 0
  total_count = 0
  for value, count in values:
    total_sum += value
    total_count += count
  yield (key, total_sum / total_count if total_count else float("nan"))

avg_result = list(MapReduce(RECORDREADER, MAP, REDUCE))
avg_result


[('avg', 13.0)]

### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [23]:
def groupbykey_sort(iterable):
  sorted_items = sorted(iterable, key=lambda x: x[0])
  if not sorted_items:
    return []

  result = []
  current_key = sorted_items[0][0]
  current_values = []
  for key, value in sorted_items:
    if key != current_key:
      result.append((current_key, current_values))
      current_key = key
      current_values = [value]
    else:
      current_values.append(value)
  result.append((current_key, current_values))
  return result

example = [
    ("b", 3),
    ("a", 1),
    ("b", 7),
    ("a", 2),
    ("c", 5),
    ("b", 9)
]

groupbykey_sort(example)


[('a', [1, 2]), ('b', [3, 7, 9]), ('c', [5])]

### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [24]:
values = [5, 1, 2, 5, 3, 1, 4, 2, 2, 7, 4, 7, 8]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for idx, value in enumerate(split):
      yield (idx, value)

  split_size = int(np.ceil(len(values) / maps))
  for i in range(0, len(values), split_size):
    yield RECORDREADER(values[i:i+split_size])

def MAP(_, value):
  yield (value, None)

def REDUCE(value, _):
  yield (value, value)

distinct_values = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE)
distinct_values = sorted([v for (_, partition) in distinct_values for (_, v) in partition])
distinct_values


13 key-value pairs were sent over a network.


[1, 2, 3, 4, 5, 7, 8]

#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [25]:
relation_R = [
    ("Анна", "math", 91),
    ("Борис", "math", 67),
    ("Вера", "physics", 84),
    ("Глеб", "math", 80),
    ("Даша", "physics", 73),
]

def RECORDREADER():
  for idx, row in enumerate(relation_R):
    yield (idx, row)

def MAP(_, t):
  if t[2] >= 80:
    yield (t, t)

def REDUCE(t, values):
  for value in values:
    yield (t, value)

selection_result = list(MapReduce(RECORDREADER, MAP, REDUCE))
selection_result


[(('Анна', 'math', 91), ('Анна', 'math', 91)),
 (('Вера', 'physics', 84), ('Вера', 'physics', 84)),
 (('Глеб', 'math', 80), ('Глеб', 'math', 80))]

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [26]:
relation_R = [
    ("Анна", "math", 91),
    ("Борис", "math", 67),
    ("Анна", "math", 95),
    ("Вера", "physics", 84),
]

def RECORDREADER():
  for idx, row in enumerate(relation_R):
    yield (idx, row)

def MAP(_, t):
  t_proj = (t[0], t[1])
  yield (t_proj, t_proj)

def REDUCE(t_proj, values):
  yield (t_proj, t_proj)

projection_result = list(MapReduce(RECORDREADER, MAP, REDUCE))
projection_result


[(('Анна', 'math'), ('Анна', 'math')),
 (('Борис', 'math'), ('Борис', 'math')),
 (('Вера', 'physics'), ('Вера', 'physics'))]

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [27]:
R = [("Анна", "math"), ("Борис", "math"), ("Вера", "physics")]
S = [("Борис", "math"), ("Глеб", "biology")]

def RECORDREADER():
  for row in R:
    yield ("R", row)
  for row in S:
    yield ("S", row)

def MAP(_, t):
  yield (t, t)

def REDUCE(t, values):
  yield (t, t)

union_result = list(MapReduce(RECORDREADER, MAP, REDUCE))
union_result


[(('Анна', 'math'), ('Анна', 'math')),
 (('Борис', 'math'), ('Борис', 'math')),
 (('Вера', 'physics'), ('Вера', 'physics')),
 (('Глеб', 'biology'), ('Глеб', 'biology'))]

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [28]:
R = [("Анна", "math"), ("Борис", "math"), ("Вера", "physics")]
S = [("Борис", "math"), ("Глеб", "biology"), ("Вера", "physics")]

def RECORDREADER():
  for row in R:
    yield ("R", row)
  for row in S:
    yield ("S", row)

def MAP(_, t):
  yield (t, t)

def REDUCE(t, values):
  if len(values) >= 2:
    yield (t, t)

intersection_result = list(MapReduce(RECORDREADER, MAP, REDUCE))
intersection_result


[(('Борис', 'math'), ('Борис', 'math')),
 (('Вера', 'physics'), ('Вера', 'physics'))]

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [29]:
R = [("Анна", "math"), ("Борис", "math"), ("Вера", "physics")]
S = [("Борис", "math"), ("Глеб", "biology")]

def RECORDREADER():
  for row in R:
    yield ("R", row)
  for row in S:
    yield ("S", row)

def MAP(relation_name, t):
  yield (t, relation_name)

def REDUCE(t, relation_names):
  relation_names = list(relation_names)
  if relation_names == ["R"]:
    yield (t, t)

difference_result = list(MapReduce(RECORDREADER, MAP, REDUCE))
difference_result


[(('Анна', 'math'), ('Анна', 'math')),
 (('Вера', 'physics'), ('Вера', 'physics'))]

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [30]:
R = [("x", 1), ("y", 2), ("z", 1)]
S = [(1, "red"), (1, "blue"), (3, "green")]

def RECORDREADER():
  for a, b in R:
    yield ("R", (a, b))
  for b, c in S:
    yield ("S", (b, c))

def MAP(relation_name, t):
  if relation_name == "R":
    a, b = t
    yield (b, ("R", a))
  else:
    b, c = t
    yield (b, ("S", c))

def REDUCE(b, values):
  left = []
  right = []
  for relation_name, value in values:
    if relation_name == "R":
      left.append(value)
    else:
      right.append(value)
  for a in left:
    for c in right:
      yield (None, (a, b, c))

join_result = list(MapReduce(RECORDREADER, MAP, REDUCE))
join_result


[(None, ('x', 1, 'red')),
 (None, ('x', 1, 'blue')),
 (None, ('z', 1, 'red')),
 (None, ('z', 1, 'blue'))]

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [31]:
sales = [
    ("shop_1", 120, "A"),
    ("shop_2", 90, "B"),
    ("shop_1", 30, "A"),
    ("shop_2", 40, "C"),
    ("shop_1", 50, "B"),
]

def RECORDREADER():
  for idx, row in enumerate(sales):
    yield (idx, row)

def MAP(_, t):
  a, b, c = t
  yield (a, b)

def REDUCE(a, values):
  yield (a, sum(values))

aggregation_result = list(MapReduce(RECORDREADER, MAP, REDUCE))
aggregation_result


[('shop_1', 200), ('shop_2', 130)]

# 

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [32]:
mat = np.array([
    [1., 2., 3.],
    [4., 5., 6.]
])
vec = np.array([10., 20., 30.])

records = []
for i in range(mat.shape[0]):
  for j in range(mat.shape[1]):
    records.append(("M", (i, j, mat[i, j])))
for j in range(vec.shape[0]):
  records.append(("V", (j, vec[j])))

def RECORDREADER():
  for idx, record in enumerate(records):
    yield (idx, record)

def MAP(_, record):
  tag, payload = record
  if tag == "M":
    i, j, mij = payload
    yield (j, ("M", i, mij))
  else:
    j, vj = payload
    yield (j, ("V", vj))

def REDUCE(j, values):
  matrix_entries = []
  vector_values = []
  for value in values:
    if value[0] == "M":
      _, i, mij = value
      matrix_entries.append((i, mij))
    else:
      _, vj = value
      vector_values.append(vj)
  for i, mij in matrix_entries:
    for vj in vector_values:
      yield (i, mij * vj)

stage1 = list(MapReduce(RECORDREADER, MAP, REDUCE))
stage1

def RECORDREADER_STAGE2():
  for i, partial_product in stage1:
    yield (i, partial_product)

def MAP_STAGE2(i, partial_product):
  yield (i, partial_product)

def REDUCE_STAGE2(i, values):
  yield (i, sum(values))

matrix_vector_result = list(MapReduce(RECORDREADER_STAGE2, MAP_STAGE2, REDUCE_STAGE2))
matrix_vector_result, np.matmul(mat, vec)


([(0, np.float64(140.0)), (1, np.float64(320.0))], array([140., 320.]))

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$. 





In [33]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))


Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [34]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):
    yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
  (i, k) = key
  yield ((i, k), sum(values))


Проверьте своё решение

In [35]:
reference_solution = np.matmul(small_mat, big_mat) 
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution))


True

In [36]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)


1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [37]:
I, J, K = 3, 4, 2
M = np.random.rand(I, J)
N = np.random.rand(J, K)

all_records = []
for i in range(M.shape[0]):
  for j in range(M.shape[1]):
    all_records.append(("M", (i, j, M[i, j])))
for j in range(N.shape[0]):
  for k in range(N.shape[1]):
    all_records.append(("N", (j, k, N[j, k])))

def RECORDREADER():
  for idx, record in enumerate(all_records):
    yield (idx, record)

def MAP(_, record):
  tag, payload = record
  if tag == "M":
    i, j, mij = payload
    yield (j, ("M", i, mij))
  else:
    j, k, njk = payload
    yield (j, ("N", k, njk))

def REDUCE(j, values):
  left = []
  right = []
  for value in values:
    if value[0] == "M":
      _, i, mij = value
      left.append((i, mij))
    else:
      _, k, njk = value
      right.append((k, njk))
  for i, mij in left:
    for k, njk in right:
      yield ((i, k), mij * njk)

stage1 = list(MapReduce(RECORDREADER, MAP, REDUCE))

def RECORDREADER_STAGE2():
  for key, value in stage1:
    yield (key, value)

def MAP_STAGE2(key, value):
  yield (key, value)

def REDUCE_STAGE2(key, values):
  yield (key, sum(values))

product_records = list(MapReduce(RECORDREADER_STAGE2, MAP_STAGE2, REDUCE_STAGE2))

def asmatrix_from_records(records):
  rows = max(i for ((i, k), v) in records) + 1
  cols = max(k for ((i, k), v) in records) + 1
  out = np.zeros((rows, cols))
  for (i, k), v in records:
    out[i, k] = v
  return out

asmatrix_from_records(product_records), np.matmul(M, N), np.allclose(asmatrix_from_records(product_records), np.matmul(M, N))


(array([[0.98411666, 0.76216045],
        [1.14519996, 0.82763395],
        [1.53690505, 1.3189159 ]]),
 array([[0.98411666, 0.76216045],
        [1.14519996, 0.82763395],
        [1.53690505, 1.3189159 ]]),
 True)

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER. 

In [38]:
maps = 2
reducers = 3

I, J, K = 3, 4, 5
M = np.random.rand(I, J)
N = np.random.rand(J, K)

def INPUTFORMAT_STAGE1():
  def RECORDREADER_M():
    for i in range(M.shape[0]):
      for j in range(M.shape[1]):
        yield ((i, j), ("M", M[i, j]))

  def RECORDREADER_N():
    for j in range(N.shape[0]):
      for k in range(N.shape[1]):
        yield ((j, k), ("N", N[j, k]))

  yield RECORDREADER_M()
  yield RECORDREADER_N()

def MAP_STAGE1(k1, v1):
  tag = v1[0]
  val = v1[1]
  if tag == "M":
    i, j = k1
    yield (j, ("M", i, val))
  else:
    j, k = k1
    yield (j, ("N", k, val))

def REDUCE_STAGE1(j, values):
  left = []
  right = []
  for value in values:
    if value[0] == "M":
      _, i, mij = value
      left.append((i, mij))
    else:
      _, k, njk = value
      right.append((k, njk))
  for i, mij in left:
    for k, njk in right:
      yield ((i, k), mij * njk)

stage1_partitioned = MapReduceDistributed(INPUTFORMAT_STAGE1, MAP_STAGE1, REDUCE_STAGE1)
stage1_records = [item for (_, partition) in stage1_partitioned for item in partition]

def INPUTFORMAT_STAGE2():
  chunk_size = int(np.ceil(len(stage1_records) / maps))
  def make_reader(chunk):
    def RECORDREADER():
      for idx, item in enumerate(chunk):
        yield (idx, item)
    return RECORDREADER()

  for start in range(0, len(stage1_records), chunk_size):
    yield make_reader(stage1_records[start:start+chunk_size])

def MAP_STAGE2(_, item):
  key, value = item
  yield (key, value)

def REDUCE_STAGE2(key, values):
  yield (key, sum(values))

stage2_partitioned = MapReduceDistributed(INPUTFORMAT_STAGE2, MAP_STAGE2, REDUCE_STAGE2)
distributed_product_records = [item for (_, partition) in stage2_partitioned for item in partition]

distributed_product = asmatrix_from_records(distributed_product_records)
distributed_product, np.allclose(distributed_product, np.matmul(M, N))


32 key-value pairs were sent over a network.
60 key-value pairs were sent over a network.


(array([[1.22575864, 1.08792645, 0.92774927, 1.2044932 , 1.04812293],
        [1.73323154, 1.10664771, 1.21780135, 1.47158689, 1.37996745],
        [1.29075639, 0.90498541, 0.89105157, 1.12617843, 1.05741663]]),
 True)

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [39]:
maps = 4
reducers = 3

I, J, K = 4, 3, 4
M = np.random.rand(I, J)
N = np.random.rand(J, K)

def split_indices(n, parts):
  chunk = int(np.ceil(n / parts))
  return [range(start, min(start + chunk, n)) for start in range(0, n, chunk)]

row_splits_M = split_indices(M.shape[0], 2)
row_splits_N = split_indices(N.shape[0], 2)

def INPUTFORMAT_STAGE1_MULTI():
  for rows in row_splits_M:
    def reader_m(rows=rows):
      for i in rows:
        for j in range(M.shape[1]):
          yield ((i, j), ("M", M[i, j]))
    yield reader_m()

  for rows in row_splits_N:
    def reader_n(rows=rows):
      for j in rows:
        for k in range(N.shape[1]):
          yield ((j, k), ("N", N[j, k]))
    yield reader_n()

stage1_partitioned = MapReduceDistributed(INPUTFORMAT_STAGE1_MULTI, MAP_STAGE1, REDUCE_STAGE1)
stage1_records = [item for (_, partition) in stage1_partitioned for item in partition]

def INPUTFORMAT_STAGE2_MULTI():
  chunk_size = int(np.ceil(len(stage1_records) / maps))
  for start in range(0, len(stage1_records), chunk_size):
    chunk = stage1_records[start:start+chunk_size]
    def reader(chunk=chunk):
      for idx, item in enumerate(chunk):
        yield (idx, item)
    yield reader()

stage2_partitioned = MapReduceDistributed(INPUTFORMAT_STAGE2_MULTI, MAP_STAGE2, REDUCE_STAGE2)
multi_reader_product_records = [item for (_, partition) in stage2_partitioned for item in partition]
multi_reader_product = asmatrix_from_records(multi_reader_product_records)

print(np.allclose(multi_reader_product, np.matmul(M, N)))
print("Корректно только если все элементы матриц покрыты полностью и без потерь.")
print("Если RECORDREADER-ы выдают случайное подмножество элементов, результат в общем случае будет неверным,")
print("потому что часть слагаемых p_(i,k) = sum_j m_(i,j) * n_(j,k) просто не попадёт в вычисление.")


24 key-value pairs were sent over a network.
48 key-value pairs were sent over a network.
True
Корректно только если все элементы матриц покрыты полностью и без потерь.
Если RECORDREADER-ы выдают случайное подмножество элементов, результат в общем случае будет неверным,
потому что часть слагаемых p_(i,k) = sum_j m_(i,j) * n_(j,k) просто не попадёт в вычисление.
